# mcp

> the vault as MCP tools any client can drive

In [ ]:
#| default_exp mcp

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import os, sys
from vishalakshi.core import Vault, KINDS

try: from mcp.server.fastmcp import FastMCP
except ImportError as e:
    raise ImportError("vishalakshi-mcp needs the `mcp` package — install with "
                      "`pip install 'vishalakshi[mcp]'`") from e

In [ ]:
#| export
mcp = FastMCP('vishalakshi', instructions=(
    'A personal research vault: web pages, papers, transcripts, files, code and notes in one '
    'searchable corpus. `context` is the main tool — it returns whole sections plus what they '
    'connect to, which is what you want before answering a question. `search` is for locating '
    'things, `read_section` for pulling one section in full, `related_sections` for "what else '
    'reads like this". Everything named add_* pulls new material in; `add_note` writes your own '
    'conclusions back so they are searched alongside the sources. Run `build_graph` after a batch '
    'of adds to enable the associative retrieval leg.'))

_V = {'vault': None}

def _vault() -> Vault:
    'The one vault this server serves, opened on first use.'
    if _V['vault'] is None:
        _V['vault'] = Vault(os.getenv('VISHALAKSHI_VAULT') or None,
                            offline=bool(os.getenv('VISHALAKSHI_OFFLINE')))
    return _V['vault']

def _trunc(s, n):
    s = s or ''
    return s if not n or len(s) <= n else s[:n] + f'\n…[truncated {len(s)-n} of {len(s)} chars]'

def _kinds(kind): return None if not kind else [k.strip() for k in kind.split(',') if k.strip()]

In [ ]:
#| export
@mcp.tool()
def status() -> dict:
    "Vault contents: document/section/chunk counts, a breakdown by kind, and which encoder is active."
    return _vault().stats()

@mcp.tool()
def search(query:str, limit:int=10, kind:str|None=None, max_chars:int=600) -> list:
    "Find chunks across the whole vault (keyword + vector, fused). kind filters to a comma-separated subset of: web,pdf,arxiv,youtube,file,code,note."
    return [{'node_id': h.get('node_id'), 'breadcrumb': h.get('breadcrumb'), 'page': h.get('page'),
             'text': _trunc(h.get('content'), max_chars)}
            for h in _vault().find(query, limit=limit, kind=_kinds(kind))]

@mcp.tool()
def context(question:str, sections:int=6, related:int=6, kind:str|None=None, max_chars:int=3000) -> dict:
    "The retrieval to read before answering a question: whole sections with provenance, plus sections reached by the entity graph and by embedding similarity. Prefer this over `search` when you intend to answer, not just locate."
    c = _vault().context(question, sections=sections, related=related, kind=_kinds(kind))
    return {'question': question, 'encoder': c.encoder,
            'results': [{'n': i, 'node_id': r.node_id, 'breadcrumb': r.breadcrumb,
                         'source': r.filename, 'pages': list(r.pages) if r.pages else None,
                         'text': _trunc(r.text, max_chars)} for i, r in enumerate(c.results, 1)],
            'related': [{'node_id': r.node_id, 'breadcrumb': r.breadcrumb, 'via': r.via}
                        for r in c.related]}

@mcp.tool()
def ask(question:str, model:str|None=None, sections:int=6, kind:str|None=None) -> dict:
    "Answer a question from the vault with a local or hosted model, citing sections. Returns the answer plus the node_ids behind each [n] so you can verify any claim with read_section."
    r = _vault().ask(question, model=model, sections=sections, kind=_kinds(kind))
    return {'question': r.question, 'answer': r.answer, 'model': r.model, 'cited': r.cited}

@mcp.tool()
def read_section(node_id:str, max_chars:int=8000) -> dict:
    "Read one section of the vault in full, reassembled from its chunks."
    s = _vault().read(node_id, max_chars=max_chars)
    return {'node_id': node_id, 'title': s.get('title'), 'text': _trunc(s.get('text'), max_chars)}

@mcp.tool()
def related_sections(node_id:str, limit:int=8) -> list:
    "What else in the vault reads like this section — nearest neighbours over the stored vectors."
    return _vault().related(node_id, limit=limit)

@mcp.tool()
def toc() -> list:
    "The table of contents of every document in the vault: titles, sections and their node_ids."
    return _vault().toc()

@mcp.tool()
def topics(min_count:int=2) -> dict:
    "Cluster the vault into labelled topics — the shape of what has been collected."
    c = _vault().map(min_count=min_count)
    return {'method': c.method, 'note': c.note,
            'clusters': [{'label': cl.label, 'size': cl.size} for cl in c.clusters]}

@mcp.tool()
def sources(kind:str|None=None) -> list:
    "Every document in the vault with its provenance — where it came from, when, and which query found it."
    return _vault().sources(kind=_kinds(kind))

In [ ]:
#| export
@mcp.tool()
def add_url(url:str, title:str|None=None, sel:str|None=None) -> dict:
    "Fetch one web page and file it in the vault (escalates past bot walls automatically). sel is a CSS selector to skip nav and ads."
    return _vault().url(url, title=title, sel=sel)

@mcp.tool()
def add_web_search(query:str, n:int=5, google:bool=False) -> dict:
    "Search the web, read the top n results, and file them all in the vault. The query is kept in each document's provenance."
    return _vault().web(query, n=n, google=google)

@mcp.tool()
def add_arxiv(id_or_url:str) -> dict:
    "Read an arXiv paper (metadata plus full text) into the vault."
    return _vault().arxiv(id_or_url)

@mcp.tool()
def add_youtube(url:str) -> dict:
    "Read a YouTube video's transcript and metadata into the vault."
    return _vault().youtube(url)

@mcp.tool()
def add_file(path:str, title:str|None=None) -> dict:
    "File one local document (PDF, markdown, text, notebook) into the vault."
    return _vault().add_file(path, title=title)

@mcp.tool()
def add_dir(path:str, types:str|None=None) -> list:
    "File every document under a local directory into the vault. Already-ingested files are skipped."
    return _vault().add_dir(path, types=types)

@mcp.tool()
def add_note(text:str, title:str|None=None, tags:list|None=None) -> dict:
    "Write a note into the vault so your own conclusions are searched alongside the sources."
    return _vault().note(text, title=title, tags=tags)

@mcp.tool()
def build_graph() -> dict:
    "(Re)build the entity graph over the vault. Run after a batch of adds — it enables the associative retrieval leg that finds sections sharing no words with the query."
    return {k: v for k, v in _vault().connect().items() if k != 'resolved'}

@mcp.tool()
def forget(doc_id:str) -> dict:
    "Remove a document, its sections and its chunks from the vault."
    _vault().forget(doc_id)
    return {'forgot': doc_id}

In [ ]:
#| export
@mcp.tool()
def index_code(path:str, graph:bool=True, env:bool=False) -> dict:
    "Point the vault at a repo and fill kosha's code store and AST call graph. Do this before code_search, symbol or where_to_add."
    return _vault().index_code(path, graph=graph, env=env)

@mcp.tool()
def code_search(query:str, limit:int=10, env:bool=True) -> list:
    "Search indexed code by meaning and by keyword. Supports key:value filters such as 'retry package:httpx' or 'lang:.py chunker'."
    return [{'mod': r.get('metadata',{}).get('mod_name'), 'path': r.get('metadata',{}).get('path'),
             'lineno': r.get('metadata',{}).get('lineno'),
             'code': _trunc(r.get('content'), 800)} for r in _vault().code_search(query, limit=limit, env=env)]

@mcp.tool()
def symbol(name:str, depth:int=1) -> dict:
    "A symbol in the call graph: its file, PageRank, degree, callers and callees. Use it to trace how code connects rather than just where a string appears."
    s = _vault().symbol(name, depth=depth)
    return {'node': s.node, 'info': {k: v for k, v in s.info.items() if k != 'co_dispatched'},
            'callers': list(s.callers)[:30], 'callees': list(s.callees)[:30]}

@mcp.tool()
def where_to_add(description:str, limit:int=5) -> list:
    "Where in the indexed repo a described change belongs, ranked over the call graph."
    return [{'mod': r.get('metadata',{}).get('mod_name') if isinstance(r, dict) else str(r)}
            for r in _vault().where_to_add(description, limit=limit)]

@mcp.tool()
def federated_search(query:str, limit:int=12, prose:bool=True, repo:bool=True, env:bool=False) -> dict:
    "One ranked list across the document vault AND the code index. Use when a question spans both — a design note and the function that implements it."
    from .code import fed_rows
    f = _vault().federate(query, limit=limit, prose=prose, repo=repo, env=env)
    return {'query': query, 'legs': f.legs, 'note': f.note, 'hits': fed_rows(f.hits)}

In [ ]:
#| export
@mcp.tool()
def find_apis(url:str, pattern:str='*', session:bool=False) -> list:
    "Discover the JSON endpoints a page calls. Listing, product and dashboard pages render from an internal API that is cleaner to read than their HTML; this finds it."
    return [dict(r) for r in _vault().apis(url, pattern=pattern, session=session)]

@mcp.tool()
def harvest_api(url:str, pattern:str='*', capture:int|None=None, pages:int=1, title:str|None=None,
                session:bool=False) -> dict:
    "Sniff a page's JSON API, pull its records, and file them in the vault as kind='data' — one searchable section per record. capture picks an endpoint from find_apis; pages>1 paginates."
    return _vault().harvest(url, pattern=pattern, capture=capture, pages=pages, title=title, session=session)

In [ ]:
#| export
@mcp.tool()
def add_watch(target:str, action:str='url', every:str='1d', note:str|None=None) -> dict:
    "Register a recurring job. action: url|web|harvest|arxiv|youtube|crawl|remind. every: '30m','6h','1d','1w'. action='remind' writes a note on a schedule with no network."
    return _vault().watch(target, action=action, every=every, note=note)

@mcp.tool()
def list_watches() -> list:
    "Every registered watch with its schedule, run count and last status."
    return _vault().watches()

@mcp.tool()
def poll_watches(limit:int|None=None) -> dict:
    "Run every watch that is due, and report what each did. This is the tick a scheduler or frontend calls."
    return _vault().poll(limit=limit)

@mcp.tool()
def unwatch(watch_id:str) -> dict:
    "Delete a watch. Documents it already filed stay in the vault."
    _vault().unwatch(watch_id)
    return {'unwatched': watch_id}

In [ ]:
#| export
def main():
    "Entry point for the `vishalakshi-mcp` console script. stdio by default; --http for Streamable HTTP."
    mcp.run(transport='streamable-http' if '--http' in sys.argv[1:] else 'stdio')